In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import joblib

from sklearn.pipeline        import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics         import classification_report, roc_auc_score

from src.data_loader         import load_raw_data, basic_profiling
from src.feature_engineering import build_viralidad, build_features, get_feature_columns
from src.preprocessing       import build_preprocessor, split_data, save_preprocessor

# IMPORTACIONES ACTUALIZADAS
from src.modeling            import get_base_models, tune_hyperparameters, cross_validate_models, train_and_save
from src.evaluation          import (
    evaluate_on_test, plot_confusion_matrices, plot_roc_curves,
    plot_feature_importance, plot_cv_comparison, plot_metrics_heatmap, save_tuning_report
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
SAMPLE_SIZE  = 100_000
RAW_PATH     = '../data/raw/global_youtube_creator_data_large.csv'
FIGURES_DIR  = '../reports/figures'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Entorno configurado correctamente.')

Entorno configurado correctamente.


In [2]:
import os, pickle
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(''))
while PROJECT_ROOT.name != 'Proyecto_Final_ML':
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
print("Working dir:", PROJECT_ROOT)

# Cargar artefactos de Fase 3
with open(PROJECT_ROOT / 'data' / 'processed' / 'fase3_artifacts.pkl', 'rb') as f:
    a3 = pickle.load(f)

X_train_proc       = a3['X_train_proc']
X_test_proc        = a3['X_test_proc']
y_test             = a3['y_test']
feature_names_proc = a3['feature_names_proc']

# Cargar artefactos de Fase 4
with open(PROJECT_ROOT / 'data' / 'processed' / 'fase4_artifacts.pkl', 'rb') as f:
    a4 = pickle.load(f)

trained_models = a4['trained_models']
best_models    = a4['best_models']
cv_results     = a4['cv_results']

# Cargar artefactos de Fase 5
with open(PROJECT_ROOT / 'data' / 'processed' / 'fase5_artifacts.pkl', 'rb') as f:
    a5 = pickle.load(f)

test_results  = a5['test_results']
mejor_modelo  = a5['mejor_modelo']
mejor_auc     = a5['mejor_auc']

print("Todo cargado para Fase 6")

Working dir: c:\Users\jerez\MachineLearningProjects\Final\Proyecto_Final_ML
Todo cargado para Fase 6


---
# FASE 6 | CONCLUSIONES Y RECOMENDACIONES
---

## 6.1 Hallazgos principales

### Variable objetivo
- La variable `viralidad` fue construida exitosamente mediante un puntaje compuesto de rango percentil ponderado de las metricas de engagement (views, likes, comments, shares).
- El dataset resulto con un balance de clases aproximadamente 50/50 (corte en la mediana), lo cual simplifica la evaluacion y evita problemas de clases desbalanceadas.

### Desempeño de modelos
- Los tres modelos superaron el umbral minimo definido en los criterios de exito.
- **Gradient Boosting** obtuvo el mejor ROC-AUC en el conjunto de prueba, seguido de **Random Forest**.
- **Regresion Logistica** sirve como referencia interpretable y muestra que existe una senal lineal en los datos.
- La brecha entre el accuracy de entrenamiento y validacion es minima en todos los modelos, indicando ausencia de sobreajuste severo.

### Caracteristicas mas importantes
- Las tasas de engagement relativo (`engagement_rate`, `likes_per_view`, `shares_per_view`) son los predictores mas fuertes.
- El `sentiment_score` y la `duracion_sec` tienen un impacto moderado.
- La `categoria` del contenido y la `region` aportan informacion contextual relevante.

## 6.2 Limitaciones

1. **Construccion de la variable objetivo**: Al usar las mismas metricas para construir `viralidad` y derivar tasas de engagement como caracteristicas, existe una correlacion inherente. Un diseno experimental con datos temporales (predecir viralidad futura con datos del pasado) seria mas riguroso.
2. **Muestra**: Se uso una muestra de 100,000 registros del dataset de 1,000,000 por eficiencia computacional. El modelo completo podria explotar mas patrones.
3. **Hiperparametros**: Se implemento RandomizedSearchCV para una optimización de los hyperparametros, sin embargo, se recomendaria en el futuro usar algoritmos de tuning bayesianos (Ejemplo, Optuna), para un mejor rendimiento.
4. **Variables externas**: Factores como tendencias del momento, algoritmo de recomendacion de YouTube y campanas de marketing no estan capturados en el dataset.

## 6.3 Recomendaciones

1. **Para creadores de contenido**: Priorizar videos con alta tasa de interaccion (likes/shares por vista) sobre el volumen bruto de vistas.
2. **Mejora del modelo**: Explorar modelos como XGBoost/LightGBM para mayor eficiencia, y aplicar SHAP values para interpretabilidad local.
3. **Validacion temporal**: Usar los primeros meses del dataset para entrenar y los ultimos para validar, simulando un escenario real de prediccion.
4. **Dashboard**: Los resultados del modelo y los hallazgos del EDA estan disponibles para su visualizacion en Power BI/Tableau a traves del archivo `data/processed/data_processed.csv`.

In [3]:
# Resumen ejecutivo final
print('      RESUMEN EJECUTIVO - PROYECTO VIRALIDAD YOUTUBE')
print(f'Dataset        : {SAMPLE_SIZE:,} videos muestreados de 1,000,000')
print(f'Variable obj   : viralidad (binaria, balance 50/50)')
print(f'Caracteristicas: {X_train_proc.shape[1]} (tras OHE y escalado)')
print(f'Particion      : 80% entrenamiento / 20% prueba (estratificada)')
print(f'Validacion     : StratifiedKFold (5 folds)')
print()
print('Metricas en conjunto de prueba:')
print(test_results[['Accuracy','F1','ROC-AUC']].round(4).to_string())
print()
print(f'Mejor modelo   : {mejor_modelo} (ROC-AUC = {mejor_auc:.4f})')


      RESUMEN EJECUTIVO - PROYECTO VIRALIDAD YOUTUBE
Dataset        : 100,000 videos muestreados de 1,000,000
Variable obj   : viralidad (binaria, balance 50/50)
Caracteristicas: 26 (tras OHE y escalado)
Particion      : 80% entrenamiento / 20% prueba (estratificada)
Validacion     : StratifiedKFold (5 folds)

Metricas en conjunto de prueba:
                     Accuracy      F1  ROC-AUC
Modelo                                        
Regresion Logistica    0.6152  0.6157   0.6654
Random Forest          0.6229  0.6599   0.6775
Gradient Boosting      0.6246  0.6577   0.6818

Mejor modelo   : Gradient Boosting (ROC-AUC = 0.6818)
